The below cell creates:

cooling_pairs_summary_k50_t250_lag_400.csv\
cooling_bootstraps_k50_t250_lag_400.npz\
event_time_per_state_da.csv

In [ ]:
from __future__ import annotations

import math
import re
from pathlib import Path

import h5py
import numpy as np
import pandas as pd


# =============================================================================
# LOCAL DIRECTORIES
# =============================================================================

# Run this notebook from the project root.
input_root = Path("./input").resolve()

# All generated preprocessing files are written here.
output_directory = Path(
    "./output/cooling-data-processing"
).resolve()


# =============================================================================
# ANALYSIS SETTINGS
# =============================================================================

k_choice = 50
tica_lag = 250
chosen_lag = 400

cooling_labels = [
    "6e9",
    "2e9",
    "6e8",
    "2e8",
    "6e7",
    "2e7",
    "6e6",
]


# =============================================================================
# COOLING BOOTSTRAP SETTINGS
# =============================================================================

cooling_bootstrap_count = 2000
cooling_bootstrap_seed = 42

dirichlet_prior_strength = 1.0
dirichlet_prior_type = "self"


# =============================================================================
# EQUILIBRIUM 230 K BOOTSTRAP SETTINGS
# =============================================================================

cold_bootstrap_count = 2000
cold_bootstrap_block_length = 500
cold_bootstrap_tail_frames = None
cold_bootstrap_seed = 123
cold_bootstrap_pseudocount = 1.0

cold_dtraj_dataset = "arr_00"


# =============================================================================
# TPM AND EVENT-TIME SETTINGS
# =============================================================================

tpm_key_pattern = r"^pi_"
tpm_row_sum_tolerance = 1.0e-8

event_time_lag_ns = 40.0


# =============================================================================
# DERIVED INPUT PATHS
# =============================================================================

state_model_directory = (
    input_root
    / "state-model"
    / f"t{tica_lag}_k{k_choice}"
)

p_samples_path = (
    state_model_directory
    / "bmsm"
    / f"lag{chosen_lag}"
    / f"P-samples-lag-{chosen_lag}.npz"
)

pairs_directory = (
    input_root
    / "cooling"
    / "pairs"
)

repex_directory = (
    state_model_directory
    / "repex-dtrajs"
)

cold_dtraj_path = (
    repex_directory
    / "230k-dtraj.h5"
)

cooling_pair_paths = {
    label: (
        pairs_directory
        / (
            f"{label}_pairs_"
            f"t{tica_lag}_"
            f"k{k_choice}.npy"
        )
    )
    for label in cooling_labels
}


# =============================================================================
# DERIVED OUTPUT PATHS
# =============================================================================

output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

cooling_csv_path = (
    output_directory
    / (
        f"cooling_pairs_summary_"
        f"k{k_choice}_"
        f"t{tica_lag}_"
        f"lag_{chosen_lag}.csv"
    )
)

bootstrap_npz_path = (
    output_directory
    / (
        f"cooling_bootstraps_"
        f"k{k_choice}_"
        f"t{tica_lag}_"
        f"lag_{chosen_lag}.npz"
    )
)

event_time_csv_path = (
    output_directory
    / "event_time_per_state_da.csv"
)


# =============================================================================
# DISPLAY EXPECTED LOCAL INPUT TREE
# =============================================================================

print("\nExpected local input files:\n")
print(f"TPM samples:\n  {p_samples_path}\n")

print("Cooling-pair files:")
for label in cooling_labels:
    print(f"  {cooling_pair_paths[label]}")

print(f"\n230 K discrete trajectory:\n  {cold_dtraj_path}\n")


# =============================================================================
# INPUT VALIDATION
# =============================================================================

required_files = [
    p_samples_path,
    cold_dtraj_path,
    *cooling_pair_paths.values(),
]

missing_files = [
    path
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required input files:\n"
        + "\n".join(
            f"  {path}"
            for path in missing_files
        )
    )


# =============================================================================
# LOAD FULL K-STATE TPM POSTERIOR SAMPLES
# =============================================================================

def load_full_tpm_samples(
    path: Path,
    *,
    expected_state_count: int,
    key_pattern: str,
    row_sum_tolerance: float,
) -> np.ndarray:
    """
    Load full posterior TPM samples with shape:

        (number_of_samples, k_choice, k_choice)

    Missing-state embedding and implicit row normalization are not used.
    """

    compiled_pattern = re.compile(
        key_pattern
    )

    blocks: list[np.ndarray] = []
    used_keys: list[str] = []

    with np.load(
        path,
        allow_pickle=False,
    ) as archive:
        for key in sorted(
            archive.files
        ):
            if not compiled_pattern.match(key):
                continue

            block = np.asarray(
                archive[key],
                dtype=float,
            )

            expected_shape = (
                expected_state_count,
                expected_state_count,
            )

            if block.ndim != 3:
                raise ValueError(
                    f"TPM array {key!r} must be three-dimensional. "
                    f"Found shape {block.shape}."
                )

            if block.shape[1:] != expected_shape:
                raise ValueError(
                    f"TPM array {key!r} has shape {block.shape}. "
                    f"Expected (samples, {expected_state_count}, "
                    f"{expected_state_count}) for k_choice="
                    f"{expected_state_count}."
                )

            blocks.append(block)
            used_keys.append(key)

    if not blocks:
        raise ValueError(
            f"No three-dimensional TPM arrays matching "
            f"{key_pattern!r} were found in:\n{path}"
        )

    samples = np.concatenate(
        blocks,
        axis=0,
    )

    if np.any(
        ~np.isfinite(samples)
    ):
        raise ValueError(
            "TPM posterior samples contain NaN or infinite values."
        )

    minimum_entry = float(
        np.min(samples)
    )

    if minimum_entry < -1.0e-12:
        raise ValueError(
            "TPM posterior samples contain negative entries. "
            f"Minimum entry: {minimum_entry:.6e}"
        )

    row_sums = samples.sum(
        axis=2
    )

    nonpositive_rows = np.argwhere(
        row_sums <= 0.0
    )

    if nonpositive_rows.size:
        sample_index, state_index = nonpositive_rows[0]

        raise ValueError(
            "TPM posterior samples contain a non-positive row sum. "
            f"First occurrence: sample={int(sample_index)}, "
            f"state={int(state_index)}."
        )

    maximum_row_sum_error = float(
        np.max(
            np.abs(
                row_sums - 1.0
            )
        )
    )

    if maximum_row_sum_error > row_sum_tolerance:
        raise ValueError(
            "TPM posterior samples are not row-stochastic within "
            "the requested tolerance.\n"
            f"Maximum |row sum - 1|: "
            f"{maximum_row_sum_error:.6e}\n"
            f"Tolerance: {row_sum_tolerance:.6e}"
        )

    print(
        f"Loaded {samples.shape[0]} posterior TPM samples "
        f"with shape ({expected_state_count}, "
        f"{expected_state_count})."
    )

    print(
        "TPM keys used: "
        + ", ".join(
            used_keys
        )
    )

    return samples


# =============================================================================
# STATIONARY DISTRIBUTION FROM ONE TPM
# =============================================================================

def stationary_distribution_from_tpm(
    transition_matrix: np.ndarray,
    *,
    eigenvalue_tolerance: float = 1.0e-8,
    negative_tolerance: float = 1.0e-10,
) -> np.ndarray:
    eigenvalues, eigenvectors = np.linalg.eig(
        transition_matrix.T
    )

    selected_index = int(
        np.argmin(
            np.abs(
                eigenvalues - 1.0
            )
        )
    )

    selected_eigenvalue = eigenvalues[
        selected_index
    ]

    if (
        abs(
            selected_eigenvalue - 1.0
        )
        > eigenvalue_tolerance
    ):
        raise ValueError(
            "Could not find an eigenvalue sufficiently close to 1. "
            f"Closest eigenvalue: {selected_eigenvalue}"
        )

    eigenvector = eigenvectors[
        :,
        selected_index,
    ]

    maximum_imaginary_component = float(
        np.max(
            np.abs(
                np.imag(eigenvector)
            )
        )
    )

    if maximum_imaginary_component > 1.0e-8:
        raise ValueError(
            "Stationary eigenvector has a non-negligible "
            "imaginary component."
        )

    stationary = np.real(
        eigenvector
    ).astype(
        float,
        copy=False,
    )

    if float(
        stationary.sum()
    ) < 0.0:
        stationary = -stationary

    minimum_population = float(
        stationary.min()
    )

    if minimum_population < -negative_tolerance:
        raise ValueError(
            "Stationary eigenvector contains negative components. "
            f"Minimum component: {minimum_population:.6e}"
        )

    stationary[
        stationary < 0.0
    ] = 0.0

    total = float(
        stationary.sum()
    )

    if (
        not np.isfinite(total)
        or total <= 0.0
    ):
        raise ValueError(
            "Stationary distribution has a non-positive "
            "or non-finite total."
        )

    return stationary / total


# =============================================================================
# LOAD TPM SAMPLES AND CALCULATE 277 K POPULATION SAMPLES
# =============================================================================

P_samples = load_full_tpm_samples(
    p_samples_path,
    expected_state_count=k_choice,
    key_pattern=tpm_key_pattern,
    row_sum_tolerance=tpm_row_sum_tolerance,
)

pi_samples = np.empty(
    (
        P_samples.shape[0],
        k_choice,
    ),
    dtype=float,
)

for sample_index, transition_matrix in enumerate(
    P_samples
):
    pi_samples[
        sample_index
    ] = stationary_distribution_from_tpm(
        transition_matrix
    )

    if (
        sample_index == 0
        or (sample_index + 1) % 250 == 0
        or sample_index + 1 == P_samples.shape[0]
    ):
        print(
            "Stationary distributions: "
            f"{sample_index + 1}/"
            f"{P_samples.shape[0]}"
        )

pi_mean = pi_samples.mean(
    axis=0
)

pi_ci = np.percentile(
    pi_samples,
    [
        2.5,
        97.5,
    ],
    axis=0,
)


# =============================================================================
# COOLING-PAIR FUNCTIONS
# =============================================================================

def load_cooling_pairs(
    path: Path,
    *,
    expected_state_count: int,
) -> tuple[np.ndarray, np.ndarray]:
    pair_array = np.load(
        path
    )

    if (
        pair_array.ndim != 2
        or pair_array.shape[1] != 2
    ):
        raise ValueError(
            f"Expected an N x 2 cooling-pair array in:\n"
            f"{path}\n"
            f"Found shape: {pair_array.shape}"
        )

    starts = pair_array[
        :,
        0,
    ].astype(
        np.int64,
        copy=False,
    )

    ends = pair_array[
        :,
        1,
    ].astype(
        np.int64,
        copy=False,
    )

    invalid_rows = (
        (starts < 0)
        | (starts >= expected_state_count)
        | (ends < 0)
        | (ends >= expected_state_count)
    )

    if np.any(invalid_rows):
        bad_indices = np.where(
            invalid_rows
        )[0][
            :10
        ]

        raise ValueError(
            f"Cooling pairs in {path} contain state IDs outside "
            f"0..{expected_state_count - 1}.\n"
            f"First invalid rows: {bad_indices.tolist()}"
        )

    return starts, ends


def build_transition_counts(
    starts: np.ndarray,
    ends: np.ndarray,
    *,
    state_count: int,
) -> np.ndarray:
    counts = np.zeros(
        (
            state_count,
            state_count,
        ),
        dtype=np.int64,
    )

    np.add.at(
        counts,
        (
            starts,
            ends,
        ),
        1,
    )

    return counts


def sample_supported_transition_row(
    counts_row: np.ndarray,
    *,
    source_state: int,
    state_count: int,
    prior_strength: float,
    prior_type: str,
    rng: np.random.Generator,
) -> np.ndarray:
    if counts_row.sum() == 0:
        sampled_row = np.zeros(
            state_count,
            dtype=float,
        )

        sampled_row[
            source_state
        ] = 1.0

        return sampled_row

    support = counts_row > 0

    # Retain the self-transition in the sampled support.
    support[
        source_state
    ] = True

    if prior_type == "self":
        prior = np.zeros(
            state_count,
            dtype=float,
        )

        prior[
            source_state
        ] = 1.0

    elif prior_type == "uniform":
        prior = np.full(
            state_count,
            1.0 / state_count,
            dtype=float,
        )

    else:
        raise ValueError(
            f"Unknown Dirichlet prior type: {prior_type!r}"
        )

    alpha_vector = (
        counts_row[
            support
        ].astype(
            float
        )
        + prior_strength
        * prior[
            support
        ]
    )

    if np.any(
        alpha_vector <= 0.0
    ):
        raise ValueError(
            "Dirichlet alpha vector contains non-positive entries."
        )

    sampled_values = rng.dirichlet(
        alpha_vector
    )

    sampled_row = np.zeros(
        state_count,
        dtype=float,
    )

    sampled_row[
        support
    ] = sampled_values

    return sampled_row


# =============================================================================
# BOOTSTRAP COOLED DISTRIBUTIONS
# =============================================================================

cooling_rng = np.random.default_rng(
    cooling_bootstrap_seed
)

cooled_samples: dict[str, np.ndarray] = {}
cooled_mean: dict[str, np.ndarray] = {}
cooled_ci: dict[str, np.ndarray] = {}

for label in cooling_labels:
    pair_path = cooling_pair_paths[
        label
    ]

    print(
        f"\nProcessing cooling label {label}"
    )

    starts, ends = load_cooling_pairs(
        pair_path,
        expected_state_count=k_choice,
    )

    transition_counts = build_transition_counts(
        starts,
        ends,
        state_count=k_choice,
    )

    label_samples = np.empty(
        (
            cooling_bootstrap_count,
            k_choice,
        ),
        dtype=float,
    )

    for bootstrap_index in range(
        cooling_bootstrap_count
    ):
        equilibrium_sample_index = int(
            cooling_rng.integers(
                0,
                pi_samples.shape[0],
            )
        )

        initial_population = pi_samples[
            equilibrium_sample_index
        ]

        cooled_population = np.zeros(
            k_choice,
            dtype=float,
        )

        for source_state in range(
            k_choice
        ):
            sampled_row = sample_supported_transition_row(
                transition_counts[
                    source_state
                ],
                source_state=source_state,
                state_count=k_choice,
                prior_strength=dirichlet_prior_strength,
                prior_type=dirichlet_prior_type,
                rng=cooling_rng,
            )

            cooled_population += (
                initial_population[
                    source_state
                ]
                * sampled_row
            )

        population_sum = float(
            cooled_population.sum()
        )

        if not np.isclose(
            population_sum,
            1.0,
            atol=1.0e-10,
            rtol=1.0e-10,
        ):
            raise RuntimeError(
                f"Cooling bootstrap population does not sum to 1 "
                f"for label {label}, sample {bootstrap_index}. "
                f"Sum={population_sum:.16f}"
            )

        label_samples[
            bootstrap_index
        ] = cooled_population

        if (
            (bootstrap_index + 1) % 500 == 0
            or bootstrap_index + 1
            == cooling_bootstrap_count
        ):
            print(
                f"  bootstrap "
                f"{bootstrap_index + 1}/"
                f"{cooling_bootstrap_count}"
            )

    cooled_samples[
        label
    ] = label_samples

    cooled_mean[
        label
    ] = label_samples.mean(
        axis=0
    )

    cooled_ci[
        label
    ] = np.percentile(
        label_samples,
        [
            2.5,
            97.5,
        ],
        axis=0,
    )

    print(
        f"  mean population sum: "
        f"{cooled_mean[label].sum():.12f}"
    )


# =============================================================================
# LOAD 230 K DISCRETE TRAJECTORY
# =============================================================================

def load_hdf5_dataset(
    path: Path,
    dataset_name: str,
) -> np.ndarray:
    with h5py.File(
        path,
        "r",
    ) as handle:
        if dataset_name not in handle:
            raise KeyError(
                f"Dataset {dataset_name!r} was not found in:\n"
                f"{path}\n"
                f"Available datasets: {list(handle.keys())}"
            )

        return np.asarray(
            handle[
                dataset_name
            ][:]
        )


# =============================================================================
# BLOCK BOOTSTRAP 230 K POPULATION
# =============================================================================

def block_bootstrap_population(
    dtraj: np.ndarray,
    *,
    state_count: int,
    bootstrap_count: int,
    block_length: int,
    tail_frames: int | None,
    rng_seed: int,
    pseudocount: float,
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
]:
    trajectory = np.asarray(
        dtraj
    ).reshape(
        -1
    )

    if trajectory.size == 0:
        raise ValueError(
            "The 230 K discrete trajectory is empty."
        )

    if tail_frames is not None:
        tail_frames = int(
            tail_frames
        )

        if (
            tail_frames <= 0
            or tail_frames > trajectory.size
        ):
            raise ValueError(
                f"tail_frames={tail_frames} is invalid for "
                f"a trajectory containing {trajectory.size} frames."
            )

        trajectory = trajectory[
            -tail_frames:
        ]

    trajectory = trajectory.astype(
        np.int64,
        copy=False,
    )

    invalid_states = (
        (trajectory < 0)
        | (trajectory >= state_count)
    )

    if np.any(invalid_states):
        bad_indices = np.where(
            invalid_states
        )[0][
            :10
        ]

        raise ValueError(
            "The 230 K discrete trajectory contains invalid "
            "state indices.\n"
            f"Indices: {bad_indices.tolist()}\n"
            f"Values: {trajectory[bad_indices].tolist()}\n"
            f"Valid range: 0..{state_count - 1}"
        )

    frame_count = int(
        trajectory.size
    )

    if (
        block_length <= 0
        or block_length > frame_count
    ):
        raise ValueError(
            f"block_length={block_length} is invalid for "
            f"{frame_count} trajectory frames."
        )

    blocks_per_sample = int(
        math.ceil(
            frame_count
            / block_length
        )
    )

    maximum_start = (
        frame_count
        - block_length
    )

    global_counts = np.bincount(
        trajectory,
        minlength=state_count,
    ).astype(
        float
    )

    visited_states = global_counts > 0.0

    rng = np.random.default_rng(
        rng_seed
    )

    population_samples = np.zeros(
        (
            bootstrap_count,
            state_count,
        ),
        dtype=float,
    )

    for bootstrap_index in range(
        bootstrap_count
    ):
        sampled_blocks = []

        for _ in range(
            blocks_per_sample
        ):
            block_start = int(
                rng.integers(
                    0,
                    maximum_start + 1,
                )
            )

            sampled_blocks.append(
                np.arange(
                    block_start,
                    block_start + block_length,
                    dtype=np.int64,
                )
            )

        sampled_indices = np.concatenate(
            sampled_blocks
        )[
            :frame_count
        ]

        sampled_trajectory = trajectory[
            sampled_indices
        ]

        sampled_counts = np.bincount(
            sampled_trajectory,
            minlength=state_count,
        ).astype(
            float
        )

        adjusted_counts = (
            sampled_counts
            + pseudocount
            * visited_states.astype(
                float
            )
        )

        adjusted_counts[
            ~visited_states
        ] = 0.0

        total = float(
            adjusted_counts.sum()
        )

        if total <= 0.0:
            raise RuntimeError(
                "A 230 K bootstrap sample produced "
                "zero total population."
            )

        population_samples[
            bootstrap_index
        ] = (
            adjusted_counts
            / total
        )

        if (
            (bootstrap_index + 1) % 500 == 0
            or bootstrap_index + 1
            == bootstrap_count
        ):
            print(
                "230 K bootstrap: "
                f"{bootstrap_index + 1}/"
                f"{bootstrap_count}"
            )

    mean_population = population_samples.mean(
        axis=0
    )

    confidence_interval = np.percentile(
        population_samples,
        [
            2.5,
            97.5,
        ],
        axis=0,
    )

    return (
        population_samples,
        mean_population,
        confidence_interval,
    )


cold_dtraj = load_hdf5_dataset(
    cold_dtraj_path,
    cold_dtraj_dataset,
)

(
    pi_230k_samples,
    pi_230k,
    pi_230k_ci,
) = block_bootstrap_population(
    cold_dtraj,
    state_count=k_choice,
    bootstrap_count=cold_bootstrap_count,
    block_length=cold_bootstrap_block_length,
    tail_frames=cold_bootstrap_tail_frames,
    rng_seed=cold_bootstrap_seed,
    pseudocount=cold_bootstrap_pseudocount,
)


# =============================================================================
# WRITE BOOTSTRAP NPZ
# =============================================================================

bootstrap_payload: dict[str, np.ndarray] = {
    "pi_samples": pi_samples,
    "pi_230K": pi_230k,
    "pi_230K_ci_95": pi_230k_ci,
    "pi_230K_samples": pi_230k_samples,
    "k_choice": np.array(
        k_choice,
        dtype=np.int64,
    ),
    "t_choice": np.array(
        tica_lag,
        dtype=np.int64,
    ),
    "chosen_lag": np.array(
        chosen_lag,
        dtype=np.int64,
    ),
    "rng_seed": np.array(
        cooling_bootstrap_seed,
        dtype=np.int64,
    ),
    "n_boot": np.array(
        cooling_bootstrap_count,
        dtype=np.int64,
    ),
    "labels": np.array(
        cooling_labels,
        dtype=object,
    ),
    "pi_source_mode": np.array(
        "tpm",
        dtype=object,
    ),
}

for label in cooling_labels:
    bootstrap_payload[
        f"cooled_{label}"
    ] = cooled_samples[
        label
    ]

np.savez_compressed(
    bootstrap_npz_path,
    **bootstrap_payload,
)


# =============================================================================
# WRITE COOLING SUMMARY CSV
# =============================================================================

summary_rows: list[dict[str, float | int | str]] = []

for label in cooling_labels:
    quartiles = np.percentile(
        cooled_samples[
            label
        ],
        [
            25.0,
            50.0,
            75.0,
        ],
        axis=0,
    )

    for state_index in range(
        k_choice
    ):
        summary_rows.append(
            {
                "state": state_index,
                "lab": label,
                "mean": float(
                    cooled_mean[
                        label
                    ][
                        state_index
                    ]
                ),
                "q25": float(
                    quartiles[
                        0,
                        state_index,
                    ]
                ),
                "median": float(
                    quartiles[
                        1,
                        state_index,
                    ]
                ),
                "q75": float(
                    quartiles[
                        2,
                        state_index,
                    ]
                ),
                "ci_lower_95": float(
                    cooled_ci[
                        label
                    ][
                        0,
                        state_index,
                    ]
                ),
                "ci_upper_95": float(
                    cooled_ci[
                        label
                    ][
                        1,
                        state_index,
                    ]
                ),
            }
        )


pi_quartiles = np.percentile(
    pi_samples,
    [
        25.0,
        50.0,
        75.0,
    ],
    axis=0,
)

for state_index in range(
    k_choice
):
    summary_rows.append(
        {
            "state": state_index,
            "lab": "pi_277K",
            "mean": float(
                pi_mean[
                    state_index
                ]
            ),
            "q25": float(
                pi_quartiles[
                    0,
                    state_index,
                ]
            ),
            "median": float(
                pi_quartiles[
                    1,
                    state_index,
                ]
            ),
            "q75": float(
                pi_quartiles[
                    2,
                    state_index,
                ]
            ),
            "ci_lower_95": float(
                pi_ci[
                    0,
                    state_index,
                ]
            ),
            "ci_upper_95": float(
                pi_ci[
                    1,
                    state_index,
                ]
            ),
        }
    )


pi_230k_quartiles = np.percentile(
    pi_230k_samples,
    [
        25.0,
        50.0,
        75.0,
    ],
    axis=0,
)

for state_index in range(
    k_choice
):
    summary_rows.append(
        {
            "state": state_index,
            "lab": "pi_230K",
            "mean": float(
                pi_230k[
                    state_index
                ]
            ),
            "q25": float(
                pi_230k_quartiles[
                    0,
                    state_index,
                ]
            ),
            "median": float(
                pi_230k_quartiles[
                    1,
                    state_index,
                ]
            ),
            "q75": float(
                pi_230k_quartiles[
                    2,
                    state_index,
                ]
            ),
            "ci_lower_95": float(
                pi_230k_ci[
                    0,
                    state_index,
                ]
            ),
            "ci_upper_95": float(
                pi_230k_ci[
                    1,
                    state_index,
                ]
            ),
        }
    )


summary_dataframe = pd.DataFrame(
    summary_rows
)

summary_dataframe.to_csv(
    cooling_csv_path,
    index=False,
)


# =============================================================================
# CALCULATE AND WRITE DA EVENT-TIME CSV
# =============================================================================

event_time_samples = np.empty(
    (
        P_samples.shape[0],
        k_choice,
    ),
    dtype=float,
)

for sample_index in range(
    P_samples.shape[0]
):
    stationary_population = pi_samples[
        sample_index
    ]

    equilibrium_flux = (
        stationary_population[
            :,
            None,
        ]
        * P_samples[
            sample_index
        ]
    )

    # Self-transition flux is excluded.
    np.fill_diagonal(
        equilibrium_flux,
        0.0,
    )

    outgoing_flux = equilibrium_flux.sum(
        axis=1
    )

    if np.any(
        outgoing_flux <= 0.0
    ):
        bad_states = np.where(
            outgoing_flux <= 0.0
        )[0]

        raise ValueError(
            "Non-positive outgoing equilibrium flux in "
            f"TPM sample {sample_index} for states "
            f"{bad_states.tolist()}."
        )

    event_time_samples[
        sample_index
    ] = (
        event_time_lag_ns
        / outgoing_flux
    )


event_time_dataframe = pd.DataFrame(
    {
        "state": np.arange(
            k_choice,
            dtype=int,
        ),
        "event_time_median_da_ns": np.median(
            event_time_samples,
            axis=0,
        ),
        "event_time_q025_da_ns": np.quantile(
            event_time_samples,
            0.025,
            axis=0,
        ),
        "event_time_q975_da_ns": np.quantile(
            event_time_samples,
            0.975,
            axis=0,
        ),
    }
)

event_time_dataframe.to_csv(
    event_time_csv_path,
    index=False,
)


# =============================================================================
# FINAL VALIDATION
# =============================================================================

expected_outputs = [
    cooling_csv_path,
    bootstrap_npz_path,
    event_time_csv_path,
]

missing_outputs = [
    path
    for path in expected_outputs
    if not path.is_file()
]

if missing_outputs:
    raise RuntimeError(
        "Expected output files were not created:\n"
        + "\n".join(
            f"  {path}"
            for path in missing_outputs
        )
    )

print(
    "\nCreated analysis outputs:"
)

for path in expected_outputs:
    print(
        f"  {path.name}\n"
        f"    {path}\n"
        f"    {path.stat().st_size / 1024**2:.3f} MiB"
    )